# pdf2learn on Colab

Convert any PDF into a self-contained, learning-optimized HTML document.

**Runtime**: pick a GPU runtime (Runtime → Change runtime type → T4 GPU) for faster marker-pdf extraction. CPU works but is slower.

Steps: install → upload a PDF → run → download the output folder as a zip.

## 1. Clone and install

In [ ]:
!git clone https://github.com/3Lafi/pdf2learn.git
%cd pdf2learn
!pip install -q -e .

### Already cloned? Pull latest
Skip on a fresh session. Run this after I push new commits to GitHub.

In [ ]:
%cd /content/pdf2learn
!git pull
!pip install -q -e .

## 2. Upload your PDFs

Two options — pick one:

**A. Upload a zip of a course folder** (recommended for `D:\COCĺĺ\course`): on Windows, right-click the folder → *Send to → Compressed (zipped) folder*, then upload that zip below.

**B. Upload individual PDF files.**

In [ ]:
from google.colab import files
import zipfile, pathlib, shutil

INPUT_DIR = pathlib.Path('inputs')
if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)
INPUT_DIR.mkdir()

uploaded = files.upload()  # pick .zip OR one or more .pdf files

for name, data in uploaded.items():
    dest = INPUT_DIR / name
    dest.write_bytes(data)
    if name.lower().endswith('.zip'):
        with zipfile.ZipFile(dest) as zf:
            zf.extractall(INPUT_DIR)
        dest.unlink()

pdfs = sorted(INPUT_DIR.rglob('*.pdf'))
print(f'Found {len(pdfs)} PDF(s):')
for p in pdfs: print(' ', p)

## 3. Convert

Runs `pdf2learn` recursively over the `inputs/` tree. Previous `output/` is wiped first so you get a clean run.

First run downloads marker-pdf model weights (~1–2 GB). Subsequent runs in the same session are fast.

In [ ]:
import shutil, subprocess, time, pathlib
from tqdm.auto import tqdm

OUT = pathlib.Path('output')
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir()

pdfs = sorted(pathlib.Path('inputs').rglob('*.pdf'))
print(f'Converting {len(pdfs)} PDF(s) one by one...')

failures = []
start = time.time()
for pdf in tqdm(pdfs, desc='pdf2learn', unit='pdf'):
    t0 = time.time()
    result = subprocess.run(
        ['pdf2learn', str(pdf), '--output-dir', 'output', '--force', '--quiet'],
        capture_output=True, text=True,
    )
    dt = time.time() - t0
    status = 'OK' if result.returncode == 0 else f'FAIL({result.returncode})'
    tqdm.write(f'  [{status}] {pdf.name}  ({dt:.1f}s)')
    if result.returncode != 0:
        failures.append((pdf, result.returncode, result.stderr.strip()))

total = time.time() - start
print()
print(f'Done: {len(pdfs) - len(failures)}/{len(pdfs)} OK  |  total {total/60:.1f} min')
for pdf, rc, err in failures:
    print(f'  FAIL rc={rc}  {pdf}:  {err[:120]}')

## 4. Preview the HTML (inline)

In [ ]:
from pathlib import Path
from IPython.display import IFrame, display
for idx in Path('output').glob('*/index.html'):
    print(idx)
    display(IFrame(str(idx), width='100%', height=600))

## 5. Download as zip

In [ ]:
!zip -qr output.zip output
from google.colab import files
files.download('output.zip')